# AlphaGo simplificado — Entrenamiento (policy + value)

Notebook listo para Google Colab (**Runtime → Change runtime type → GPU**).

Requiere el repositorio subido/descomprimido (zip con `engine/ ai/ training/ data/historical/ requirements-train.txt`) en una de las rutas detectadas por la primera celda.

Pipelines:
1. Carga el dataset (SGF 9×9 de `data/historical/`). Si está vacío, genera partidas de self-play aleatorio como semilla.
2. Entrena la CNN compartida (`training/nets.py`).
3. Exporta `policy.onnx` + `value.onnx` y verifica la inferencia del motor.
4. Self-play: la IA juega contra sí misma y genera las partidas de la siguiente versión (v2, v3).

In [ ]:
from google.colab import files
from pathlib import Path
import os

zip_ruta = Path('/content/AlphaGo_colab.zip')

# ---- Elige cómo subir el zip (descomenta la que uses) ----
# (A) selector de archivos locales → pulsa, elige AlphaGo_colab.zip:
# uploaded = files.upload()
# (B) o bien si ya está en /content/ subido con Files ▶ Upload:
# (nada; se detecta directamente)
if not zip_ruta.exists():
    uploaded = files.upload()  # (A) abre el selector local

if zip_ruta.exists():
    print('Descomprimiendo', zip_ruta.name)
    os.system('unzip -q -o /content/AlphaGo_colab.zip -d /content/')
else:
    print('No se subió el zip: uso el repositorio ya presente en el entorno.')

In [ ]:
import sys
from pathlib import Path

# Configura aquí si el repositorio está en otra ruta; con None se auto-detecta.
RUTA_BASE = None

if RUTA_BASE is None:
    candidatos = [
        Path('/content/AlphaGo'),
        Path('/content/drive/MyDrive/AlphaGo'),
        Path.cwd(),
    ]
    for c in candidatos:
        if (c / 'training' / 'dataset.py').exists():
            RUTA_BASE = c
            break

assert RUTA_BASE is not None, (
    'No encontré el repositorio: configura RUTA_BASE en esta celda')
RUTA_BASE = Path(RUTA_BASE)
if str(RUTA_BASE) not in sys.path:
    sys.path.insert(0, str(RUTA_BASE))
print('RUTA_BASE =', RUTA_BASE)

In [ ]:
!pip -q install torch onnxruntime numpy
print('dependencias listas')

In [ ]:
import random
import torch
import numpy as np

from training.dataset import cargar_directorio, resumen_texto
from training.nets import RedesAlphaGo, entrenar, exportar_onnx
from training.self_play import generar_muchas

torch.manual_seed(42)
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print('dispositivo:', dispositivo)

## 1. Dataset

Cada muestra: canales del tablero (5), etiqueta de política (0..81, 82 = pase) y etiqueta de valor (1 = gana quien mueve, cuando el resultado se conoce).

In [ ]:
datos = cargar_directorio(RUTA_BASE / 'data' / 'historical')
if not datos['muestras']:
    print('Dataset histórico vacío: genero SGF de self-play aleatorio como semilla...')
    generar_muchas(partidas=8, config='aleatorio',
                   directorio=RUTA_BASE / 'data' / 'historical')
    datos = cargar_directorio(RUTA_BASE / 'data' / 'historical')

muestras = datos['muestras']
random.Random(42).shuffle(muestras)
corte = int(len(muestras) * 0.9)
entrenamiento, validacion = muestras[:corte], muestras[corte:]
print(f'{len(muestras)} muestras · tren {len(entrenamiento)} · val {len(validacion)}')

## 2. Entrenamiento

Loss = CrossEntropy(política) + BCE(valor sobre posiciones con resultado conocido). Ajusta `epochs` si el dataset es muy grande/pequeño.

In [ ]:
modelo = RedesAlphaGo(tamano=9, canales_conv=32)
modelo.to(dispositivo)
print('parámetros:', sum(p.numel() for p in modelo.parameters()))

historial = entrenar(modelo, entrenamiento, epochs=30, tamano_lote=128,
                     lr=1e-3, verbose=True)
print('train finalizado')

## 3. Export + verificación

Genera `models/v<N>/policy.onnx` y `models/v<N>/value.onnx`, y comprueba que `ai/redes.py` los lee.

In [ ]:
version = 'v1'  # v2/v3 tras cada iteración de self-play
dir_modelos = RUTA_BASE / 'models' / version
exportar_onnx(modelo.to('cpu'), dir_modelos)
print('exportados:', sorted(str(p.name) for p in dir_modelos.glob('*.onnx')))

# Copiamos al directorio raíz models/ (donde espera cargar_redes())
dir_raiz = RUTA_BASE / 'models'
dir_raiz.mkdir(parents=True, exist_ok=True)
for nombre in ('policy.onnx', 'value.onnx'):
    origen = dir_modelos / nombre
    if origen.exists():
        (dir_raiz / nombre).write_bytes(origen.read_bytes())
print('en models/ raíz:', sorted(str(p.name) for p in dir_raiz.glob('policy.onnx')))

In [ ]:
from ai.redes import cargar_redes
from engine.scoring import Partida

redes = cargar_redes(dir_modelos)
assert redes is not None, 'no se cargaron los modelos ONNX'

partida0 = Partida(9, 7.5)
movs, probs = redes.distribucion_politica(partida0.tablero, 1)
print('top jugadas (inicio):', sorted(probs.items(), key=lambda kv: -kv[1])[:4])
print('valor estimado (negro):', round(redes.estimar_valor(partida0.tablero, 1), 3))
print('✓ modelos operativos')

## 4. Self-play v1 → v2

La IA (MCTS con las redes recién entrenadas) juega contra sí misma; los SGF van a `data/training/v<N>/` y alimentan el reentrenamiento de la siguiente versión. En el MCTS Python, `mcts-200+red` es razonable para explorar; sube simulaciones para más calidad.

In [ ]:
dir_selfplay = RUTA_BASE / 'data' / 'training' / version
rutas = generar_muchas(partidas=5, config='mcts-200+red', directorio=dir_selfplay,
                       semilla_inicial=1)
print(f'{len(rutas)} partidas SGF en {dir_selfplay}')
print('siguiente iteración: reentrenar con historical + training y exportar como v2')

## Qué traer de vuelta al repo local

1. `models/v1/policy.onnx` y `models/v1/value.onnx` → `AlphaGo/models/v1/`
2. (opcional) `data/training/v1/*.sgf` → para iterar a v2 en la siguiente ejecución.